# Lecture 2 — Image Classification

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 2.  
**Merged with:** Stanford CS231n 2026 [Lecture 2: *Image Classification with Linear Classifiers*](https://cs231n.stanford.edu/slides/2026/lecture_2.pdf) (the k-nearest-neighbor half; the linear-classifier half is in the Lecture 3 notebook, matching UMich).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture (its slides couldn't be read directly); the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there; **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**, with an explanation before it and a "**What to notice**" note after the important results.
* Every function has a comment saying **what it computes, what goes in, and what comes out**.
* New ideas start with a **tiny example you can check by hand**, then move to real data.
* **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**The data.** We use the **UCI handwritten digits** that come with scikit-learn: 1,797 real 8×8 grayscale images of the digits 0–9. They're small enough that everything runs in seconds, and real enough to show every idea in the lecture. A PyTorch cell at the end runs the same method on **CIFAR-10** (in Colab).

| § | Topic | What you'll build |
|---|---|---|
| 1 | The image classification task; the **semantic gap** | What the computer sees |
| 2 | Why it's hard: viewpoint, lighting, occlusion, … | How far small changes move an image |
| 3 | Why classification matters | — |
| 4 | Trying to hand-code a classifier | A rule-based digit classifier |
| 5 | The **data-driven approach**; datasets | Dataset sizes |
| 6 | **Nearest neighbor** | The classifier, from a 4×4 example up; fast distance computation |
| 7 | 🔎 Fast approximate search | A FAISS-style index |
| 8 | **k-nearest neighbors**; L1 vs. L2 distance | Decision regions, voting, ties |
| 9 | **Hyperparameters** and how to choose them | All four ideas, measured; cross-validation |
| 10 | Universal approximation; the **curse of dimensionality** | Approximation and distance experiments |
| 11 | Why kNN on raw pixels fails | Different changes, identical distances |
| 12 | Better features | Pixels vs. features vs. data augmentation |
| 13 | 🔎 Evaluating a classifier properly | Confusion matrix, top-k, duplicate detection |
| 14 | PyTorch on CIFAR-10 | (Colab) |

### Setup
* **NumPy** for arrays, **Matplotlib** for plots.
* **SciPy**: `ndimage` (shifting, rotating, blurring images) and `cdist` (fast distance computations).
* **scikit-learn**: only to load the digits dataset.

In [ ]:
import numpy as np                               # arrays and math
import matplotlib.pyplot as plt                  # plots
from scipy import ndimage                        # shift / rotate / blur images
from scipy.spatial.distance import cdist         # fast pairwise distances
from sklearn.datasets import load_digits         # the handwritten-digits dataset
import time, math, collections

np.set_printoptions(precision=3, suppress=True)

**Load the data.** `digits.images` holds 1,797 images of 8×8 pixels with values 0 (white) to 16 (black ink); `digits.target` holds the correct label (0–9) of each.

In [ ]:
digits = load_digits()
images = digits.images.astype(np.float64)        # (1797, 8, 8)
labels = digits.target                           # (1797,)
print("images:", images.shape, " labels:", labels.shape, " pixel values from", images.min(), "to", images.max())

fig, ax = plt.subplots(1, 10, figsize=(12, 1.6))
for d in range(10):
    ax[d].imshow(images[np.nonzero(labels == d)[0][0]], cmap="gray_r"); ax[d].set_title(str(d)); ax[d].axis("off")
plt.suptitle("one example of each digit", fontsize=9); plt.show()

**Flatten and split.** Most methods in this lecture treat an image as one long vector: an 8×8 image becomes **64 numbers**. We then split the data at random into a **training set** (to learn from) and a **test set** (to measure performance on images the method has never seen). Shuffling first matters: the file is roughly sorted, and we don't want all of one kind of image on one side of the split.

In [ ]:
X_all = images.reshape(len(images), -1)          # (1797, 64): each row is one flattened image
split_rng = np.random.default_rng(0)
perm = split_rng.permutation(len(X_all))          # a random order of the indices 0..1796
train_idx, test_idx = perm[:1300], perm[1300:]
X_train, y_train = X_all[train_idx], labels[train_idx]
X_test, y_test = X_all[test_idx], labels[test_idx]
print(f"train: {X_train.shape}, test: {X_test.shape}")
print("training images per class:", np.bincount(y_train))

---
## 1. Image classification: a core task in computer vision

* **Input:** an image.
* **Output:** one label from a **fixed set of possible labels**, e.g., {dog, cat, truck, plane, …}.

Simple to state, and the foundation of almost everything else in this course.

### 1.1 The problem: the semantic gap
We see a cat. The computer sees **a grid of numbers**: an 800×600 color image is 800×600×3 = 1.44 million integers between 0 and 255. The distance between that grid and the idea "cat" is the **semantic gap**. Here's one digit, as we see it and as the computer sees it:

In [ ]:
i = 0
fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
ax[0].imshow(images[i], cmap="gray_r"); ax[0].set_title(f"what we see: a '{labels[i]}'"); ax[0].axis("off")
ax[1].imshow(np.zeros((8, 8)), cmap="gray_r", vmin=0, vmax=1)
for r in range(8):
    for c in range(8):
        ax[1].text(c, r, int(images[i, r, c]), ha="center", va="center", fontsize=9)   # write each pixel's value
ax[1].set_title("what the computer sees"); ax[1].set_xticks([]); ax[1].set_yticks([])
plt.show()

---
## 2. Challenges: why those numbers are hard to interpret

Each of these changes the pixel values a lot, while the label stays the same:
* **Viewpoint variation:** *all pixels change when the camera moves!*
* **Illumination:** bright light, shadow, colored light.
* **Background clutter:** the object blends into a busy background.
* **Occlusion:** only part of the object is visible.
* **Deformation:** the object bends into different poses (cats are famously flexible).
* **Intraclass variation:** many different-looking things share one label (cats of every breed).
* **Fine-grained categories** (UMich): telling very similar classes apart (different cat breeds).
* **Context** **[CS231n 2026]**: surroundings change what we think we see, and models can over-rely on them (recognizing "cow" mostly from green grass).

**Let's measure it.** We need a way to say how "far apart" two images are. The simplest is the **Euclidean (L2) distance** between their pixel vectors: subtract pixel by pixel, square, add up, and take the square root:
$$d_2(I_1, I_2) = \sqrt{\sum_p\left(I_1^p - I_2^p\right)^2}$$
Tiny example first: two 2-pixel "images", (1, 2) and (4, 6), are $\sqrt{3^2 + 4^2} = 5$ apart.

In [ ]:
def l2_distance(a, b):
    # Euclidean distance between two images (any shape): sqrt of the sum of squared pixel differences.
    a, b = np.ravel(np.asarray(a, float)), np.ravel(np.asarray(b, float))   # flatten, so any shapes with equal size work
    return float(np.sqrt(((a - b) ** 2).sum()))

print("distance between (1, 2) and (4, 6):", l2_distance([1, 2], [4, 6]))

Now take one digit "0" and make four slightly changed versions of it, the kind of change a human wouldn't even notice:

In [ ]:
def small_changes(x):
    # Four small, label-preserving changes of an 8x8 image x. Returns a dict {description: changed image}.
    occluded = x.copy(); occluded[4:6, :] = 0                                # blank out two rows (an "occluder")
    return {
        "shifted 1 pixel right": ndimage.shift(x, (0, 1), order=0, cval=0),
        "darker (x0.6)": 0.6 * x,
        "occluded (2 rows)": occluded,
        "rotated 15 degrees": ndimage.rotate(x, 15, reshape=False, order=1),
    }

x0 = images[0]                                                               # a '0'
changes = small_changes(x0)
fig, ax = plt.subplots(1, 5, figsize=(10, 2.2))
for a, (name, im) in zip(ax, [("original", x0)] + list(changes.items())):
    a.imshow(im, cmap="gray_r", vmin=0, vmax=16); a.set_title(name, fontsize=8); a.axis("off")
plt.show()

How far did each change move the image, and how does that compare with the **closest image of a different digit** in the whole dataset?

In [ ]:
others = X_all[labels != labels[0]]                                          # every image that is NOT a '0'
closest_other = others[np.argmin(((others - x0.ravel()) ** 2).sum(1))]
print(f"distance to the most similar DIFFERENT digit: {l2_distance(x0, closest_other):.1f}")
for name, im in changes.items():
    print(f"distance to the same '0', {name:22s}: {l2_distance(x0, im):.1f}")

**What to notice:** a one-pixel shift moves the "0" **farther** than the most similar image of a *different* digit. Lighting, occlusion, and rotation move it about as far. **Pixel space is not organized by meaning**: that's the core difficulty of image classification.

---
## 3. Image classification is very useful, and a building block

**Directly useful:** classifying medical scans, galaxies (Dieleman et al. 2014), individual whales from photos (a Kaggle competition), and many industrial tasks. **A building block for other tasks:**
* **Object detection:** classify many candidate regions of an image (*background, cat, dog, …*) (Lectures 15–16).
* **Image captioning:** write a caption one word at a time; each step **classifies** which word comes next (Lecture 12).
* **Playing Go:** the board is a 19×19 "image"; choosing a move is a **classification** over 361 positions (AlphaGo).

---
## 4. Can we just write an image classifier by hand?

```python
def classify_image(image):
    # Some magic here?
    return class_label
```
Sorting a list has an obvious algorithm; **recognizing a cat doesn't.** People tried: find edges and corners (Lecture 1), then write rules about how they're arranged. Let's try for digits, using a sensible shape cue: the **number of holes** (closed loops). 0, 6, and 9 have one hole; 8 has two; 1, 2, 3, 5, 7 have none.

**Step 1: count holes.** A hole is a region of background (no ink) that **doesn't touch the image border**. `ndimage.label` finds connected regions; we keep the background regions that are enclosed. We upsample the 8×8 image 4× first so thin strokes close their loops.

In [ ]:
def holes_and_positions(im):
    # Count enclosed background regions ("holes") in a digit image.
    # Returns (number of holes, vertical position of each hole from 0 = top to 1 = bottom, ink width from 0 to 1).
    ink = ndimage.zoom(im, 4, order=1) > 6                      # 32x32 boolean: True where there's ink
    background, n_regions = ndimage.label(~ink)                 # connected regions of NON-ink
    holes = []
    for lab in range(1, n_regions + 1):
        region = background == lab
        touches_border = region[0].any() or region[-1].any() or region[:, 0].any() or region[:, -1].any()
        if not touches_border and region.sum() > 4:             # enclosed and not a tiny speck
            holes.append(lab)
    hole_rows = [np.nonzero(background == lab)[0].mean() / ink.shape[0] for lab in holes]
    ink_width = ink.any(0).sum() / ink.shape[1]                 # fraction of columns containing any ink
    return len(holes), hole_rows, ink_width

for d in [0, 1, 8, 6]:
    k = np.nonzero(labels == d)[0][0]
    n, rows, width = holes_and_positions(images[k])
    print(f"a '{d}': {n} hole(s) at height {np.round(rows, 2).tolist()}, ink width {width:.2f}")

**Notice the "6":** its loop isn't fully closed at this resolution, so it has **no** detected hole, even though every 6 "should" have one. That's the problem with hand-written cues in a nutshell: real data keeps breaking the assumptions behind them.

**Step 2: write rules** using those cues anyway, and measure them on the test set.

In [ ]:
def rule_based_classifier(im):
    # Hand-written rules based on holes and width. Returns a digit guess.
    n_holes, hole_rows, width = holes_and_positions(im)
    if n_holes >= 2:
        return 8
    if n_holes == 1:
        r = hole_rows[0]
        return 9 if r < 0.4 else (6 if r > 0.6 else 0)        # hole near the top -> 9, bottom -> 6, middle -> 0
    if width < 0.45:
        return 1                                                # narrow ink -> 1
    return 7                                                    # ...and we've run out of good ideas

preds = np.array([rule_based_classifier(im) for im in images[test_idx]])
print(f"hand-written rules: test accuracy {np.mean(preds == y_test):.1%}  (random guessing: 10%)")
for d in [0, 1, 6, 8, 9, 2, 3]:
    print(f"   digit {d}: {np.mean(preds[y_test == d] == d):.0%} correct")

**What to notice:** a few rules work for some digits, but every rule has exceptions (an open-topped 4, a 2 with a loop), and digits without a distinctive cue (2, 3, 5, 7) get no rule at all. Hand-writing rules for *cats* would be hopeless. We need the data to define the classifier.

---
## 5. Machine learning: the data-driven approach

1. **Collect a dataset** of images and labels.
2. Use **machine learning** to **train** a classifier.
3. **Evaluate** the classifier on **new** images.

```python
def train(images, labels):
    # Machine learning!
    return model

def predict(model, test_images):
    # Use model to predict labels
    return test_labels
```
The program is now **learned from examples** instead of written by hand. Every method in this course has these two functions.

### 5.1 Image classification datasets
| Dataset | Classes | Images | Image size | Notes |
|---|---|---|---|---|
| **MNIST** | 10 digits | 50K train (+10K val), 10K test | 28×28 gray | "The *Drosophila* of computer vision"; results on MNIST often don't hold on harder datasets! |
| **CIFAR-10** | 10 (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck) | 50K train, 10K test | 32×32 color | |
| **CIFAR-100** | 100, in 20 superclasses | 50K train, 10K test | 32×32 color | e.g., *aquatic mammals*: beaver, dolphin, otter, seal, whale |
| **ImageNet** | 1,000 | ~1.3M train, 50K val, 100K test | ~256×256 | Score: top-5 accuracy |
| **MIT Places** | 365 scene types | ~8M train | 256×256 | Scenes, not objects |
| **Omniglot** | 1,623 characters from 50 alphabets | 20 per class | | **Few-shot learning**: learn a new class from a few examples |

How much data is that? Pixels in each training set = (number of images) × (height × width × channels):

In [ ]:
datasets = {"MNIST": 60_000 * 28 * 28, "CIFAR-10": 50_000 * 32 * 32 * 3, "CIFAR-100": 50_000 * 32 * 32 * 3,
            "ImageNet": 1_281_167 * 256 * 256 * 3, "Places365": 8_000_000 * 256 * 256 * 3}
for name, px in datasets.items():
    print(f"{name:10s}: {px:>20,d} pixels  (~{px / 1e9:,.2f} billion)")
plt.figure(figsize=(6, 3)); plt.bar(datasets.keys(), datasets.values()); plt.yscale("log")
plt.ylabel("pixels (log scale)"); plt.title("training-set sizes"); plt.show()

---
## 6. Our first classifier: nearest neighbor

* **`train`:** memorize all the training images and their labels.
* **`predict`:** for a new image, find the **most similar** training image and copy its label.

To define "most similar" we need a **distance metric**. The lecture uses the **L1 (Manhattan) distance**: compare the images pixel by pixel and add up the absolute differences:
$$d_1(I_1, I_2) = \sum_p\left|I_1^p - I_2^p\right|$$

**The lecture's 4×4 example** (from the CS231n materials):

In [ ]:
test_image = np.array([[56, 32, 10, 18], [90, 23, 128, 133], [24, 26, 178, 200], [2, 0, 255, 220]])
train_image = np.array([[10, 20, 24, 17], [8, 10, 89, 100], [12, 16, 178, 170], [4, 32, 233, 112]])
pixelwise = np.abs(test_image - train_image)                  # |difference| at each pixel
print("pixel-wise absolute differences:\n", pixelwise)
print("L1 distance = sum of them =", pixelwise.sum())          # the lecture's answer: 456

**The classifier.** `train` just stores the data. `predict` loops over the test images; for each one, it computes the L1 distance to **every** training image in one vectorized line (NumPy subtracts the test image from each row of the training matrix), then takes the label of the smallest distance.

In [ ]:
class NearestNeighbor:
    # 1-nearest-neighbor classifier with L1 distance.

    def train(self, X, y):
        # X: (N, D) training images, one per row; y: (N,) labels. "Training" is just memorizing.
        self.X_train, self.y_train = X, y

    def predict(self, X):
        # X: (M, D) test images. Returns (M,) predicted labels.
        predictions = np.zeros(len(X), dtype=self.y_train.dtype)
        for i in range(len(X)):
            distances = np.sum(np.abs(self.X_train - X[i]), axis=1)   # L1 distance to every training image
            predictions[i] = self.y_train[np.argmin(distances)]        # label of the closest one
        return predictions

nn = NearestNeighbor()
nn.train(X_train, y_train)
t0 = time.time(); pred = nn.predict(X_test); elapsed = time.time() - t0
print(f"1-nearest neighbor (L1): test accuracy {np.mean(pred == y_test):.1%}   ({elapsed * 1000:.0f} ms to predict {len(X_test)} images)")

**What to notice:** far better than the hand-written rules, with **no knowledge about digits at all**: the data did all the work. Let's look at what it's doing: some test images and their 5 nearest training images (green = same label as the test image, red = different):

In [ ]:
fig, ax = plt.subplots(6, 6, figsize=(6, 6))
for row in range(6):
    query = X_test[row]
    nearest = np.argsort(np.abs(X_train - query).sum(1))[:5]           # indices of the 5 closest training images
    ax[row, 0].imshow(query.reshape(8, 8), cmap="gray_r"); ax[row, 0].set_title("test", fontsize=7)
    for j, n_idx in enumerate(nearest):
        same = y_train[n_idx] == y_test[row]
        ax[row, j + 1].imshow(X_train[n_idx].reshape(8, 8), cmap="Greens" if same else "Reds")
        ax[row, j + 1].set_title(str(y_train[n_idx]), fontsize=7)
for a in ax.ravel(): a.axis("off")
plt.show()

### 6.1 How fast are training and prediction?
**With N training images: training is O(1), prediction is O(N).** (Training stores the data; predicting compares with all N images.) **That's backwards from what we want**: training happens once, offline, but prediction happens constantly, often on a phone. We want classifiers that are **fast at prediction**; slow training is fine. Neural networks are exactly that.

### 6.2 🔎 Computing many distances fast
The loop above handles one test image at a time. For the **L2** distance there's a classic trick (a standard exercise in both courses' first assignment) that computes **all** test-to-train distances with a single matrix multiplication, using
$$\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\,a^\top b$$
(Expand $(a - b)^\top(a - b)$.) Check it on two small vectors first:

In [ ]:
a, b = np.array([1.0, 2.0, 3.0]), np.array([4.0, 0.0, -1.0])
print("||a - b||^2 directly:         ", np.sum((a - b) ** 2))
print("||a||^2 + ||b||^2 - 2 a.b:     ", a @ a + b @ b - 2 * (a @ b))

Now three implementations of "all L2 distances between the rows of X and the rows of T", from slowest to fastest. In the last one, `(X**2).sum(1)[:, None]` makes a column of $\|x_i\|^2$, `(T**2).sum(1)[None, :]` a row of $\|t_j\|^2$, and NumPy **broadcasting** adds them into a full matrix.

In [ ]:
def l2_two_loops(X, T):
    # Distance matrix D[i, j] = ||X[i] - T[j]||, one pair at a time.
    D = np.zeros((len(X), len(T)))
    for i in range(len(X)):
        for j in range(len(T)):
            D[i, j] = np.sqrt(np.sum((X[i] - T[j]) ** 2))
    return D

def l2_one_loop(X, T):
    # Same, one row of X at a time (vectorized over T).
    return np.stack([np.sqrt(((T - X[i]) ** 2).sum(1)) for i in range(len(X))])

def l2_no_loops(X, T):
    # Same, with no Python loops: ||x||^2 + ||t||^2 - 2 x.t for every pair at once.
    sq = (X ** 2).sum(1)[:, None] + (T ** 2).sum(1)[None, :] - 2 * X @ T.T
    return np.sqrt(np.maximum(sq, 0))           # rounding can make tiny values slightly negative; clip before sqrt

Check they agree, and time them:

In [ ]:
Xs = X_test[:200]
times, D_ref = {}, None
for f in [l2_two_loops, l2_one_loop, l2_no_loops]:
    t0 = time.time(); D = f(Xs, X_train); times[f.__name__] = time.time() - t0
    D_ref = D if D_ref is None else D_ref
    print(f"{f.__name__:13s}: {times[f.__name__] * 1000:7.1f} ms   max difference from the two-loop result: {np.abs(D - D_ref).max():.1e}")
print(f"no loops is ~{times['l2_two_loops'] / times['l2_no_loops']:.0f}x faster than two loops")

**What to notice:** identical results, very different speed. Pushing loops into NumPy (or a GPU) is called **vectorization**, and you'll use it in every assignment. The `np.maximum(sq, 0)` matters: when two images are nearly identical, rounding can make $\|a\|^2 + \|b\|^2 - 2a^\top b$ come out slightly negative, and the square root of a negative number is `nan`.

---
## 7. 🔎 Fast (approximate) nearest-neighbor search

*Many methods exist for fast / approximate nearest-neighbor search; a good implementation is **FAISS*** (Johnson, Douze & Jégou, 2017) **[CS231n 2026]**. Approximate search powers image retrieval, recommendations, and retrieval-augmented language models, all searching billions of vectors.

**The inverted-file (IVF) idea**, FAISS's most common index:
1. **Offline:** cluster the training images into $C$ groups ("cells") with **k-means**.
2. **At query time:** find the query's nearest few cells ($n_{\text{probe}}$ of them), and compare the query only with the training images **in those cells**.

**k-means** repeats two steps: (a) assign every point to its nearest center; (b) move every center to the average of its points.

In [ ]:
def kmeans(X, k, iters=20, seed=0):
    # Cluster the rows of X into k groups. Returns (centers (k, D), assignment of each row (N,)).
    r = np.random.default_rng(seed)
    centers = X[r.choice(len(X), k, replace=False)].copy()            # start from k random data points
    for _ in range(iters):
        assign = np.argmin(l2_no_loops(X, centers), axis=1)           # (a) nearest center for every point
        for j in range(k):
            if np.any(assign == j):
                centers[j] = X[assign == j].mean(0)                    # (b) move center to the mean of its points
    return centers, np.argmin(l2_no_loops(X, centers), axis=1)

n_cells = 32
centers, cell_of = kmeans(X_train, n_cells)
cells = [np.nonzero(cell_of == j)[0] for j in range(n_cells)]          # which training images are in each cell
print(f"{n_cells} cells; images per cell: min {min(map(len, cells))}, max {max(map(len, cells))}")

**The search.** For each query, rank the cells by distance to their centers, gather the training images in the closest `n_probe` cells, and find the nearest among only those. We count how many distances that took, and compare the result with the exact nearest neighbor (**recall@1**: how often we found the true nearest neighbor).

In [ ]:
def ivf_search(q, n_probe):
    # Approximate nearest neighbor of q. Returns (index of the neighbor found, number of distances computed).
    closest_cells = np.argsort(((centers - q) ** 2).sum(1))[:n_probe]
    candidates = np.concatenate([cells[j] for j in closest_cells])
    best = candidates[np.argmin(((X_train[candidates] - q) ** 2).sum(1))]
    return best, len(candidates) + n_cells                              # + n_cells distances to rank the cells

exact = np.argmin(l2_no_loops(X_test, X_train), axis=1)
print(f"{'n_probe':>8s} {'distances/query':>16s} {'speedup':>8s} {'recall@1':>9s} {'accuracy':>9s}")
for n_probe in [1, 2, 4, 8, 32]:
    found, cost = zip(*[ivf_search(q, n_probe) for q in X_test])
    found = np.array(found)
    print(f"{n_probe:8d} {np.mean(cost):16.0f} {len(X_train) / np.mean(cost):7.1f}x "
          f"{np.mean(found == exact):9.1%} {np.mean(y_train[found] == y_test):9.1%}")

**What to notice:** checking just a couple of cells finds the true nearest neighbor most of the time, at a fraction of the cost, and classification accuracy barely changes, because a *nearly*-nearest neighbor usually has the same label.

---
## 8. k-nearest neighbors

### 8.1 What does nearest neighbor look like?
Decision boundaries are easiest to see in 2-D. Every training point "owns" the region of space closer to it than to any other point; 1-NN colors that region with the point's label. **Problem:** a single mislabeled or unusual point (an **outlier**) carves out its own island.

**k-nearest neighbors:** instead of copying the label of the single nearest neighbor, **take a majority vote among the $k$ closest points.** Larger $k$ **smooths the boundaries** and **reduces the effect of outliers**. When the vote is **tied**, the point is ambiguous; a real implementation needs a tie-breaking rule.

**Step 1: a 2-D toy dataset**: three clusters, with three deliberately mislabeled points.

In [ ]:
toy_rng = np.random.default_rng(3)
cluster_centers = np.array([[0, 0], [2.2, 0.4], [1.0, 2.0]])
P2 = np.vstack([c + toy_rng.normal(0, 0.6, (25, 2)) for c in cluster_centers])     # 25 points per cluster
Y2 = np.repeat([0, 1, 2], 25)
Y2[[3, 30, 60]] = [1, 2, 0]                                                           # three mislabeled points (outliers)
colors = np.array(["#1f4e9c", "#c0392b", "#1e8449"])
plt.figure(figsize=(4, 4)); plt.scatter(*P2.T, c=colors[Y2], s=18, edgecolors="k", linewidths=0.4)
plt.title("toy data (3 outliers)"); plt.show()

**Step 2: the voting rule**, with a tiny example. Suppose a query's 5 nearest neighbors have labels [2, 0, 2, 1, 0]: the votes are class 0: 2, class 1: 1, class 2: 2, a **tie** between 0 and 2. We mark ties with −1 so we can see them on the plot.

In [ ]:
def vote(neighbor_labels, n_classes=3):
    # Majority vote. Returns the winning class, or -1 if two or more classes tie for the most votes.
    counts = np.bincount(neighbor_labels, minlength=n_classes)
    winners = np.nonzero(counts == counts.max())[0]
    return winners[0] if len(winners) == 1 else -1

print("votes for [2, 0, 2, 1, 0]:", np.bincount([2, 0, 2, 1, 0], minlength=3), "-> result", vote(np.array([2, 0, 2, 1, 0])))
print("votes for [2, 0, 2, 1, 2]:", np.bincount([2, 0, 2, 1, 2], minlength=3), "-> result", vote(np.array([2, 0, 2, 1, 2])))

**Step 3: kNN prediction for many query points**, with either distance:

In [ ]:
def knn_predict(X_tr, y_tr, X_query, k, metric="l2"):
    # Label each query point by a vote of its k nearest training points (-1 = tie).
    D = l2_no_loops(X_query, X_tr) if metric == "l2" else cdist(X_query, X_tr, "cityblock")   # "cityblock" = L1
    nearest = np.argsort(D, axis=1)[:, :k]                     # indices of the k nearest, per query
    return np.array([vote(y_tr[row]) for row in nearest])

grid_x, grid_y = np.meshgrid(np.linspace(-2, 4, 200), np.linspace(-2, 4, 200))
grid = np.c_[grid_x.ravel(), grid_y.ravel()]                   # every pixel of the plot, as a query point
region_colors = plt.matplotlib.colors.ListedColormap(["white", "#8fb3e0", "#f4a582", "#a6dba0"])   # white = tie

fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
for a, k in zip(ax, [1, 3, 5]):
    regions = knn_predict(P2, Y2, grid, k).reshape(grid_x.shape)
    a.imshow(regions + 1, origin="lower", extent=(-2, 4, -2, 4), cmap=region_colors, vmin=0, vmax=3)
    a.scatter(*P2.T, c=colors[Y2], s=14, edgecolors="k", linewidths=0.4)
    a.set_title(f"K = {k}  (ties: {np.mean(regions == -1):.1%} of the plane)"); a.set_xticks([]); a.set_yticks([])
plt.show()

**What to notice:** with K = 1, each mislabeled point creates an island of the wrong color. With K = 3 and 5, the islands vanish and the boundaries smooth out. The white areas are ties.

### 8.2 Distance metric: L1 vs. L2
$$\text{L1 (Manhattan):}\ \ d_1(I_1, I_2) = \sum_p|I_1^p - I_2^p|\qquad\qquad\text{L2 (Euclidean):}\ \ d_2(I_1, I_2) = \sqrt{\sum_p(I_1^p - I_2^p)^2}$$
* **L2** is the straight-line distance, "as the crow flies".
* **L1** is the distance walking along grid lines, like walking city blocks **[CS231n 2026]**.

**CS231n's worked example:** from $O = (0, 0)$, the points $A = (1, 0)$ and $B = (0.5, 0.5)$ are **equally far in L1** (both 1), but not in L2. And $A$ and $B' = (1/\sqrt2, 1/\sqrt2)$ are equally far in L2, but not in L1:

In [ ]:
d1 = lambda a, b: float(np.abs(np.subtract(a, b)).sum())
d2 = lambda a, b: float(np.sqrt((np.subtract(a, b) ** 2).sum()))
O, A, B, B2 = (0, 0), (1, 0), (0.5, 0.5), (1 / np.sqrt(2), 1 / np.sqrt(2))
print(f"L1: d(O, A) = {d1(O, A):.3f}, d(O, B) = {d1(O, B):.3f}   <- equal")
print(f"L2: d(O, A) = {d2(O, A):.3f}, d(O, B) = {d2(O, B):.3f}   <- B is closer")
print(f"L2: d(O, A) = {d2(O, A):.3f}, d(O, B') = {d2(O, B2):.3f}  <- equal;  but L1: d(O, B') = {d1(O, B2):.3f}")

🔎 **One real difference: L1 depends on how you orient the axes; L2 doesn't.** Rotate two points together and their L2 distance is unchanged, but their L1 distance changes. So **L1** can make sense when each input dimension has a specific meaning (salary, age, years of experience), and **L2** is the natural default when the axes are arbitrary.

In [ ]:
theta = np.radians(30)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])   # rotation by 30 degrees
p, q = np.array([1.0, 0.2]), np.array([-0.3, 0.9])
print(f"before rotation: L2 {d2(p, q):.3f}, L1 {d1(p, q):.3f}")
print(f"after rotation:  L2 {d2(R @ p, R @ q):.3f} (same), L1 {d1(R @ p, R @ q):.3f} (changed)")

**The shapes behind it**: all points at distance 1 from the origin form a **circle** in L2 but a **diamond** in L1. That's why their decision boundaries look different:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
t = np.linspace(0, 2 * np.pi, 200)
ax[0].plot(np.cos(t), np.sin(t), label="L2 distance = 1 (circle)")
ax[0].plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], label="L1 distance = 1 (diamond)")
ax[0].set_aspect("equal"); ax[0].legend(fontsize=8)
for a, metric in zip(ax[1:], ["l1", "l2"]):
    regions = knn_predict(P2, Y2, grid, 1, metric=metric).reshape(grid_x.shape)
    a.imshow(regions + 1, origin="lower", extent=(-2, 4, -2, 4), cmap=region_colors, vmin=0, vmax=3)
    a.scatter(*P2.T, c=colors[Y2], s=14, edgecolors="k", linewidths=0.4)
    a.set_title(f"1-NN with {metric.upper()}"); a.set_xticks([]); a.set_yticks([])
plt.show()

L1 boundaries are made of horizontal, vertical, and 45° pieces; L2 boundaries are straight lines at any angle.

**With the right distance, kNN works on any kind of data**: e.g., UMich's example of finding similar research papers by comparing their text (tf-idf similarity, arxiv-sanity.com). *Try it:* http://vision.stanford.edu/teaching/cs231n-demos/knn/ **[CS231n 2026]**.

---
## 9. Hyperparameters

**What's the best $k$? The best distance metric?** These are **hyperparameters**: **choices about the learning algorithm itself, which we don't learn from the training data**; we set them beforehand. They're **very problem-dependent**: in general, try several and see what works best. But "see what works" on *which* data?

First, a kNN class for real data. **Tie rule:** if several classes tie, pick the one whose member is **closest** to the query (a common, sensible choice). Tiny example: neighbors (closest first) with labels [3, 5, 5, 3]: classes 3 and 5 tie with 2 votes each, and the closest neighbor is a 3, so we pick 3.

In [ ]:
def vote_closest_tiebreak(neighbor_labels):
    # Majority vote over labels sorted from closest to farthest; ties go to the tied class seen first (= closest).
    counts = collections.Counter(neighbor_labels)
    top = max(counts.values())
    tied = {c for c, v in counts.items() if v == top}
    return next(lab for lab in neighbor_labels if lab in tied)

print("neighbors [3, 5, 5, 3] ->", vote_closest_tiebreak([3, 5, 5, 3]))
print("neighbors [5, 3, 5, 1] ->", vote_closest_tiebreak([5, 3, 5, 1]), "(5 wins outright)")

In [ ]:
def pairwise_distances(X_query, X_ref, metric):
    # (M, N) matrix of distances between query rows and reference rows. metric: "l1", "l2", or "cosine".
    if metric == "l2":
        return l2_no_loops(X_query, X_ref)
    if metric == "l1":
        return cdist(X_query, X_ref, "cityblock")
    if metric == "cosine":                                    # 1 - cosine of the angle between the vectors
        A = X_query / (np.linalg.norm(X_query, axis=1, keepdims=True) + 1e-12)
        B = X_ref / (np.linalg.norm(X_ref, axis=1, keepdims=True) + 1e-12)
        return 1 - A @ B.T
    raise ValueError(metric)

class KNN:
    # k-nearest-neighbor classifier. Hyperparameters: k and the distance metric.
    def __init__(self, k=1, metric="l2"):
        self.k, self.metric = k, metric

    def train(self, X, y):
        self.X_train, self.y_train = X, y
        return self

    def predict(self, X, D=None):
        # D: optional precomputed distance matrix (saves time when trying several k on the same data).
        if D is None:
            D = pairwise_distances(X, self.X_train, self.metric)
        nearest = np.argsort(D, axis=1, kind="stable")[:, :self.k]          # closest first
        return np.array([vote_closest_tiebreak(list(self.y_train[row])) for row in nearest])

### 9.1 Four ideas for setting hyperparameters
* **Idea #1: choose the values that work best on the training data.** **BAD:** $k = 1$ always works perfectly on the training data.
* **Idea #2: split into train and test; choose the values that work best on the test data.** **BAD:** then we have no idea how the algorithm will do on truly new data. *Never do this!* **[CS231n 2026]**
* **Idea #3: split into train, validation, and test; choose on validation, evaluate on test.** **Better!**
* **Idea #4: cross-validation:** split the data into folds, use each fold as validation in turn, and average. Useful for small datasets, but not used much in deep learning (it means training 5 times).

**Idea #1, tested.** Why is k = 1 always perfect on training data? Each training image's nearest neighbor is **itself**, at distance 0.

In [ ]:
for k in [1, 3, 7]:
    train_acc = np.mean(KNN(k).train(X_train, y_train).predict(X_train) == y_train)
    print(f"k = {k}: accuracy on the TRAINING data {train_acc:.1%}" + ("   <- every point is its own nearest neighbor" if k == 1 else ""))

### 9.2 Why Idea #2 is a trap: measuring the cost of tuning on the test set
If you try many settings and keep the one with the best **test** accuracy, that number is **too optimistic**: you've picked the setting that happened to be lucky on those particular test images. We can measure how optimistic:
1. Define 60 settings (k, metric, and a random subset of pixels to use).
2. Split the data into training, a small **test set A** (used to pick the best setting), and a large **test set B** (never used for choosing).
3. Report the winner's accuracy on A and on B. Repeat over 20 random splits.

**Step 1: the settings.**

In [ ]:
settings_rng = np.random.default_rng(0)
settings = [(k, m, settings_rng.random(64) < 0.6)                  # (k, metric, which pixels to use)
            for k in [1, 3, 5, 9] for m in ["l1", "l2", "cosine"] for _ in range(5)]
print(f"{len(settings)} settings, e.g. k = {settings[0][0]}, metric = {settings[0][1]}, using {settings[0][2].sum()} of 64 pixels")

**Step 2: the experiment.**

In [ ]:
def tune_on_A_check_on_B(split_seed):
    # Pick the best setting on test set A; return (its accuracy on A, its accuracy on fresh test set B).
    p = np.random.default_rng(100 + split_seed).permutation(len(X_all))
    tr, A, B = p[:500], p[500:650], p[650:]
    acc_A, acc_B = [], []
    for k, metric, mask in settings:
        model = KNN(k, metric).train(X_all[tr][:, mask], labels[tr])
        acc_A.append(np.mean(model.predict(X_all[A][:, mask]) == labels[A]))
        acc_B.append(np.mean(model.predict(X_all[B][:, mask]) == labels[B]))
    best = int(np.argmax(acc_A))
    return acc_A[best], acc_B[best]

results = np.array([tune_on_A_check_on_B(s) for s in range(20)])
print(f"accuracy reported on test set A (used for choosing): {results[:, 0].mean():.1%}")
print(f"the same setting on fresh test set B:                {results[:, 1].mean():.1%}")
print(f"optimism from tuning on the test set: {100 * (results[:, 0] - results[:, 1]).mean():+.1f} points (average of 20 splits)")

**What to notice:** the number you'd report after tuning on the test set is higher than what the model actually achieves on new data. More settings or a smaller test set make it worse. That's why the test set is used **once, at the very end**. (At the scale of a whole field: Recht et al. 2019 built a new ImageNet test set and found accuracies dropped by 11–14 points, though the ranking of models was mostly preserved.)

### 9.3 Ideas #3 and #4: validation, and 5-fold cross-validation
**5-fold cross-validation:** split the training data into 5 equal folds. For each fold in turn, train on the other 4 and measure accuracy on it. The 5 results tell you both the **average** and the **spread** (how much the estimate depends on which images land in the validation fold). Tiny illustration of the folds for 10 items:

In [ ]:
items = np.arange(10)
for f, val in enumerate(np.array_split(items, 5)):
    print(f"fold {f}: validate on {val.tolist()}, train on {np.setdiff1d(items, val).tolist()}")

In [ ]:
def cross_validate(X, y, ks, metric, n_folds=5, seed=0):
    # Returns an (n_folds, len(ks)) array of validation accuracies.
    folds = np.array_split(np.random.default_rng(seed).permutation(len(X)), n_folds)
    results = np.zeros((n_folds, len(ks)))
    for f in range(n_folds):
        val = folds[f]
        tr = np.concatenate([folds[g] for g in range(n_folds) if g != f])
        D = pairwise_distances(X[val], X[tr], metric)          # compute distances once per fold, reuse for every k
        for j, k in enumerate(ks):
            results[f, j] = np.mean(KNN(k, metric).train(X[tr], y[tr]).predict(X[val], D=D) == y[val])
    return results

CS231n's version uses CIFAR-10 (*each point is a single outcome; the line goes through the mean; bars show the standard deviation; k ≈ 7 works best for that data*). Ours, on the digits, for two metrics:

In [ ]:
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31]
cv = {}
plt.figure(figsize=(7, 4))
for metric, color in [("l1", "tab:orange"), ("l2", "tab:blue")]:
    res = cross_validate(X_train, y_train, ks, metric); cv[metric] = res
    for j, k in enumerate(ks):
        plt.scatter([k] * len(res), res[:, j], color=color, s=8, alpha=0.5)     # each fold's result
    plt.errorbar(ks, res.mean(0), yerr=res.std(0), color=color, capsize=3, label=f"{metric.upper()} (mean +/- std)")
plt.xlabel("k"); plt.ylabel("validation accuracy"); plt.legend(); plt.title("5-fold cross-validation"); plt.show()

for m in cv:
    j = int(np.argmax(cv[m].mean(0)))
    print(f"{m.upper()}: best k = {ks[j]}, mean {cv[m].mean(0)[j]:.1%}, fold std {cv[m].std(0)[j]:.1%};  "
          f"means for k = 1, 3, 5: " + ", ".join(f"{v:.1%}" for v in cv[m].mean(0)[:3]))

**Choose, then test once:**

In [ ]:
best_metric, best_k = max(((m, ks[int(np.argmax(cv[m].mean(0)))]) for m in cv), key=lambda mk: cv[mk[0]].mean(0).max())
final_model = KNN(best_k, best_metric).train(X_train, y_train)
print(f"chosen by cross-validation: k = {best_k}, {best_metric.upper()};  TEST accuracy (used once): {np.mean(final_model.predict(X_test) == y_test):.1%}")

**What to notice:** compare the **error bars** with the differences between neighboring $k$ values: they're similar in size, so the ranking of k = 1, 3, 5 is not very certain. Don't over-read small differences. The best $k$ for these clean digits is small; for CIFAR-10, where pixels are much less informative, larger $k$ helped.

---
## 10. Universal approximation, and the curse of dimensionality

### 10.1 Universal approximation
**As the number of training samples goes to infinity, nearest neighbor can represent any(\*) function!** (\*subject to technical conditions: e.g., continuous functions on a bounded domain, and assumptions about how training points are spread.) Here's **1-NN regression** (predict the value of the nearest training point) on a 1-D function, as the training set grows:

In [ ]:
true_function = lambda x: np.sin(3 * x) + 0.5 * np.cos(7 * x)
x_query = np.linspace(0, 2, 1000)
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, n in zip(ax, [5, 20, 80, 320]):
    x_train = np.sort(np.random.default_rng(1).uniform(0, 2, n)); y_train_f = true_function(x_train)
    nearest = np.argmin(np.abs(x_query[:, None] - x_train[None, :]), axis=1)        # nearest training x for each query
    prediction = y_train_f[nearest]
    a.plot(x_query, true_function(x_query), "k-", lw=1); a.plot(x_query, prediction, "r-", lw=1); a.scatter(x_train, y_train_f, s=8)
    a.set_title(f"N = {n}: mean error {np.mean(np.abs(prediction - true_function(x_query))):.3f}")
plt.show()

### 10.2 Problem: the curse of dimensionality
For **uniform coverage** of a space, the number of training points needed **grows exponentially with the dimension**: spacing points 1/4 apart takes 4 points on a line, $4^2 = 16$ in a square, $4^3 = 64$ in a cube, and $4^d$ in $d$ dimensions. Images have many dimensions:
* Possible 32×32 **binary** (black/white) images: $2^{32\times32}\approx10^{308}$.
* Elementary particles in the visible universe: about $10^{97}$.

🔎 **Two more ways to feel it:**
* **Neighborhoods stop being local.** In the unit cube $[0, 1]^d$, a smaller cube that holds 1% of uniformly spread points needs side length $0.01^{1/d}$: 0.01 in 1-D, but **0.63 in 10-D** and **0.9985 in 3,072-D** (CIFAR's dimension). Your "neighbors" come from nearly the entire range of every coordinate.
* **Distances concentrate.** In high dimensions, the distances from a point to all others become nearly equal, so "nearest" means little.

In [ ]:
for d in [1, 2, 3, 10, 64, 3072]:
    points_needed = f"{4.0 ** d:.2e}" if d < 300 else f"10^{round(d * math.log10(4))}"
    print(f"d = {d:5d}: points for spacing 1/4 = 4^d = {points_needed:>9s};  side of a cube holding 1% = {0.01 ** (1 / d):.4f}")
print(f"possible 32x32 binary images: 2^1024 = 10^{1024 * math.log10(2):.1f}")

**Distance concentration, measured:** 500 random points in $d$ dimensions; compare the nearest and farthest distances from one of them.

In [ ]:
conc_rng = np.random.default_rng(0)
for d in [2, 10, 100, 1000, 3072]:
    pts = conc_rng.random((500, d))
    dist = np.sqrt(((pts[1:] - pts[0]) ** 2).sum(1))
    print(f"d = {d:5d}: (farthest - nearest) / nearest = {(dist.max() - dist.min()) / dist.min():7.3f}")

**What to notice:** in 2-D the farthest point is dozens of times farther than the nearest; in thousands of dimensions they're within ~10%. Real images aren't uniformly random (they lie near a much lower-dimensional surface), which is why kNN works at all on digits, but this is why raw pixel distances are so weak.

---
## 11. kNN on raw pixels is never used

1. **Very slow at test time** (§6.1).
2. **Distances between pixel vectors aren't informative.** The lecture's famous example: an original image and three versions, **occluded**, **shifted by one pixel**, and **tinted**, that **all have the same L2 distance to the original**, although we'd judge them very differently.

We can reproduce it exactly. The idea: a change is a difference image $\Delta = \text{changed} - \text{original}$. Rescaling $\Delta$ keeps the *kind* of change but sets its size, so we can force each change to have exactly the same L2 distance.

In [ ]:
def with_distance(original, changed, target):
    # Keep the direction of the change (changed - original) but rescale it so its L2 length equals `target`.
    delta = changed - original
    return original + delta * (target / np.linalg.norm(delta))

x3 = images[np.nonzero(labels == 3)[0][0]]                                  # a '3'
shifted = ndimage.shift(x3, (0, 1), order=0, cval=0)                       # shift right by 1 pixel
target = np.linalg.norm(shifted - x3)                                      # use the shift's distance as the common target
box = np.zeros((8, 8), bool); box[2:5, 2:6] = True
occluded = with_distance(x3, np.where(box, 0, x3), target)                 # blank out a box, rescaled
tinted = with_distance(x3, x3 + 4.0, target)                               # brighten everything, rescaled

fig, ax = plt.subplots(1, 4, figsize=(9, 2.4))
for a, (name, im) in zip(ax, [("original", x3), ("occluded", occluded), ("shifted (1 px)", shifted), ("tinted", tinted)]):
    a.imshow(im, cmap="gray_r", vmin=0, vmax=20); a.set_title(f"{name}\nL2 = {np.linalg.norm(im - x3):.2f}", fontsize=8); a.axis("off")
plt.show()

**What to notice:** three very different changes, one identical distance. Pixel distance measures *how much the numbers changed*, not *what changed*. In practice that makes kNN on pixels fragile to exactly the variations of §2. Shift every test digit by one pixel (a change no human would notice):

In [ ]:
shift_by = lambda ims, s: np.array([ndimage.shift(im, (0, s), order=0, cval=0) for im in ims]).reshape(len(ims), -1)
X_test_shift1 = shift_by(images[test_idx], 1)
X_test_shift2 = shift_by(images[test_idx], 2)
print(f"kNN on pixels: original test images {np.mean(final_model.predict(X_test) == y_test):.1%},  "
      f"shifted 1 pixel {np.mean(final_model.predict(X_test_shift1) == y_test):.1%}")

---
## 12. Nearest neighbor with better features

The problem isn't nearest neighbor; it's comparing **raw pixels**. Compare **features** that capture what matters. We'll try three fixes against 1- and 2-pixel shifts:
* **Blurred pixels:** smooth first, so nearby pixels share information (a natural guess for shift tolerance).
* **🔎 Gradient-orientation histograms** (HOG-style, Dalal & Triggs 2005; the SIFT idea of Lecture 1): describe each small cell of the image by *which edge directions* it contains, pooled over the cell.
* **Data augmentation:** keep raw pixels, but **add shifted copies of every training image** to the training set.

With **ConvNet features**, nearest neighbor works very well: e.g., captioning a new image with the caption of its nearest training image in feature space (Devlin et al. 2015).

**Feature 1: blur.**

In [ ]:
def blur_features(flat_images):
    # Gaussian-blur each 8x8 image (sigma = 1 pixel) and flatten again.
    ims = flat_images.reshape(-1, 8, 8)
    return np.array([ndimage.gaussian_filter(im, 1.0) for im in ims]).reshape(len(ims), -1)

**Feature 2: HOG-style histograms.** For each image: upsample to 16×16, compute the gradient (direction and strength) at each pixel, split into a 4×4 grid of cells, and in each cell add up gradient strength in 8 direction bins. 4×4×8 = 128 numbers, normalized to length 1.

In [ ]:
def hog_features(flat_images, n_bins=8):
    # Gradient-orientation histograms: 4x4 cells x 8 direction bins = 128 numbers per image.
    feats = []
    for im in flat_images.reshape(-1, 8, 8):
        up = ndimage.zoom(im, 2, order=1)                                 # 16x16
        gy, gx = np.gradient(ndimage.gaussian_filter(up, 0.8))
        direction_bin = ((np.arctan2(gy, gx) % np.pi) / np.pi * n_bins).astype(int) % n_bins   # 0..7 (unsigned direction)
        strength = np.hypot(gx, gy)
        f = []
        for i in range(0, 16, 4):
            for j in range(0, 16, 4):                                     # one 4x4 cell
                f.extend(np.bincount(direction_bin[i:i + 4, j:j + 4].ravel(),
                                     weights=strength[i:i + 4, j:j + 4].ravel(), minlength=n_bins))
        f = np.array(f); feats.append(f / (np.linalg.norm(f) + 1e-9))
    return np.array(feats)

print("HOG feature vector length:", hog_features(X_train[:1]).shape[1])

**Fix 3: data augmentation.** Build a training set 5× larger: every training image shifted by −2, −1, 0, +1, +2 pixels, all keeping their labels.

In [ ]:
X_train_aug = np.vstack([shift_by(images[train_idx], s) for s in (-2, -1, 0, 1, 2)])
y_train_aug = np.tile(y_train, 5)
print("augmented training set:", X_train_aug.shape)

**Compare all four** (kNN with k = 3, L2) on original, 1-pixel-shifted, and 2-pixel-shifted test images:

In [ ]:
print(f"{'method':32s} {'original':>9s} {'shift 1px':>10s} {'shift 2px':>10s}")
for name, feat in [("raw pixels", lambda z: z), ("blurred pixels", blur_features), ("gradient histograms (HOG)", hog_features)]:
    m = KNN(3, "l2").train(feat(X_train), y_train)
    accs = [np.mean(m.predict(feat(T)) == y_test) for T in (X_test, X_test_shift1, X_test_shift2)]
    print(f"{name:32s} {accs[0]:9.1%} {accs[1]:10.1%} {accs[2]:10.1%}")
m = KNN(3, "l2").train(X_train_aug, y_train_aug)
accs = [np.mean(m.predict(T) == y_test) for T in (X_test, X_test_shift1, X_test_shift2)]
print(f"{'raw pixels + shifted copies':32s} {accs[0]:9.1%} {accs[1]:10.1%} {accs[2]:10.1%}   (5x the training data)")

**What to notice:**
* **Blurring doesn't help here.** At 8×8 resolution there's little detail to spare: it hurts on original images and doesn't fix shifts. (A plausible idea that the measurement rejects.)
* **Gradient histograms help a lot for 1-pixel shifts**, because pooling directions over a cell tolerates small moves (Lecture 1's complex cells). But a 2-pixel shift, a quarter of the digit's width, defeats every hand-designed feature here.
* **Data augmentation works best:** with shifted training copies, a shifted test digit has a similarly shifted training neighbor. **The invariance comes from the data.** The cost for kNN: 5× the memory and prediction time. Neural networks get the same benefit without growing at test time (Lectures 10–11).

Hand-designing features for cats and trucks is much harder than for digits. **The rest of the course is about learning features**, and kNN on learned features is still widely used: image retrieval, deduplication, few-shot classification, and the standard **"kNN evaluation"** of self-supervised models (e.g., DINO, Caron et al. 2021).

---
## 13. 🔎 Evaluating a classifier properly

Accuracy is one number; three more tools tell you much more.

### 13.1 The confusion matrix
Entry $(i, j)$ counts test images whose **true** label is $i$ and **predicted** label is $j$. The diagonal holds the correct predictions; off-diagonal entries show **which classes get confused**. Tiny example with 3 classes:

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes):
    # C[i, j] = number of examples with true label i predicted as j.
    C = np.zeros((n_classes, n_classes), int)
    np.add.at(C, (y_true, y_pred), 1)              # add 1 at each (true, predicted) pair
    return C

print(confusion_matrix(np.array([0, 0, 1, 1, 2, 2]), np.array([0, 1, 1, 1, 2, 0]), 3))
print("row = true class, column = prediction: one 0 was called a 1, one 2 was called a 0")

On the shifted test digits, where errors actually happen:

In [ ]:
C = confusion_matrix(y_test, final_model.predict(X_test_shift1), 10)
plt.figure(figsize=(4.5, 4)); plt.imshow(C, cmap="Blues")
for i in range(10):
    for j in range(10):
        if C[i, j]:
            plt.text(j, i, C[i, j], ha="center", va="center", fontsize=7, color="white" if C[i, j] > 25 else "black")
plt.xlabel("predicted"); plt.ylabel("true"); plt.xticks(range(10)); plt.yticks(range(10)); plt.title("confusion matrix (shifted test digits)"); plt.show()
off = C.copy(); np.fill_diagonal(off, 0)
i_, j_ = np.unravel_index(off.argmax(), off.shape)
print(f"most common confusion: true {i_} predicted as {j_} ({off[i_, j_]} times);  per-class accuracy ranges "
      f"from {(C.diagonal() / C.sum(1)).min():.0%} to {(C.diagonal() / C.sum(1)).max():.0%}")

### 13.2 Top-k accuracy
A prediction counts as correct if the true class is **among the $k$ highest-scoring classes** (ImageNet reports top-5). For kNN, a natural score for each class is its **number of votes** among, say, the 10 nearest neighbors. Tiny example: votes [0, 6, 0, 3, 1, 0, 0, 0, 0, 0] rank class 1 first and class 3 second, so if the truth is 3, it's a top-1 miss but a top-2 hit.

In [ ]:
def topk_accuracy(X_tr, y_tr, X_te, y_te, n_neighbors=10, ks=(1, 2, 3)):
    # Rank classes by votes among the n nearest neighbors (ties broken toward closer neighbors); report top-k accuracy.
    nearest = np.argsort(l2_no_loops(X_te, X_tr), axis=1)[:, :n_neighbors]
    neighbor_labels = y_tr[nearest]
    votes = np.stack([(neighbor_labels == c).sum(1) for c in range(10)], 1).astype(float)
    first_position = np.stack([np.argmax(neighbor_labels == c, 1) for c in range(10)], 1)   # where each class first appears
    scores = votes - 1e-3 * first_position                                                  # tie-break: closer wins
    ranked = np.argsort(-scores, axis=1)
    return {k: np.mean([y in ranked[i, :k] for i, y in enumerate(y_te)]) for k in ks}

print("top-k accuracy on SHIFTED test digits:", {f"top-{k}": f"{v:.1%}" for k, v in topk_accuracy(X_train, y_train, X_test_shift1, y_test).items()})

### 13.3 Train/test duplicates
A test score only means something if the test images are **new**. Datasets collected from the web often have near-duplicates across the split: Barz & Denzler (2020) found near-duplicates of about **3.3% of CIFAR-10 and 10% of CIFAR-100 test images** in the training sets (and released cleaned test sets, "ciFAIR"). Nearest-neighbor search is the tool for finding them: a duplicate is unusually close to some training image.

**Experiment:** plant 30 near-duplicates (training images plus a little noise) in our test set, and flag every test image whose nearest training image is closer than 99% of genuine test images are.

In [ ]:
dup_rng = np.random.default_rng(7)
dup_sources = dup_rng.choice(len(X_train), 30, replace=False)
planted = X_train[dup_sources] + dup_rng.normal(0, 0.5, (30, 64))           # near-copies of training images
X_audit = np.vstack([X_test, planted])
is_duplicate = np.r_[np.zeros(len(X_test), bool), np.ones(30, bool)]

nearest_train_distance = l2_no_loops(X_audit, X_train).min(axis=1)
threshold = np.quantile(nearest_train_distance[~is_duplicate], 0.01)        # closer than 99% of genuine test images
flagged = nearest_train_distance < threshold
print(f"median distance to the nearest training image: genuine {np.median(nearest_train_distance[~is_duplicate]):.1f}, "
      f"planted {np.median(nearest_train_distance[is_duplicate]):.1f}")
print(f"flagged {flagged.sum()}: caught {np.sum(flagged & is_duplicate)} of 30 planted duplicates, {np.sum(flagged & ~is_duplicate)} false alarms")
print(f"1-NN accuracy on the duplicates: {np.mean(KNN(1).train(X_train, y_train).predict(planted) == y_train[dup_sources]):.0%} "
      "(they inflate test accuracy: the model has seen the answers)")

In real projects, run this check (on learned embeddings, at scale with FAISS) before trusting a test number. Wrong labels matter too: Northcutt et al. (2021) estimated about 6% of ImageNet's validation labels are wrong.

---
## 14. PyTorch: kNN on CIFAR-10 (run in Colab)

The same method on the real CIFAR-10 benchmark, on a GPU. The CS231n course notes report **38.6% (L1)** and **35.4% (L2)** test accuracy for 1-NN on raw pixels with the full training set (vs. 10% chance, and well over 90% for a modern CNN). `torch.cdist` computes all pairwise distances; `topk(..., largest=False)` finds the k smallest; `torch.mode` takes the majority vote.

In [ ]:
# [PyTorch] Needs: pip install torch torchvision; downloads CIFAR-10 (~170MB). Use a GPU runtime.
import torch, torchvision
device = "cuda" if torch.cuda.is_available() else "cpu"
train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
Xtr = torch.tensor(train_set.data, dtype=torch.float32, device=device).reshape(50000, -1) / 255   # (50000, 3072)
ytr = torch.tensor(train_set.targets, device=device)
Xte = torch.tensor(test_set.data[:2000], dtype=torch.float32, device=device).reshape(2000, -1) / 255
yte = torch.tensor(test_set.targets[:2000], device=device)

def knn_torch(X_train, y_train, X_query, k):
    # kNN with L2 distance on the GPU: all distances, the k nearest, then a majority vote.
    distances = torch.cdist(X_query, X_train)                       # (queries, train)
    nearest = distances.topk(k, largest=False).indices              # (queries, k)
    return torch.mode(y_train[nearest], dim=1).values               # most common label (ties -> smallest label)

for k in [1, 3, 7, 15]:
    acc = (knn_torch(Xtr, ytr, Xte, k) == yte).float().mean().item()
    print(f"CIFAR-10, k = {k:2d}: test accuracy on 2,000 images {acc:.1%}")

---
## 15. Summary

| Topic | Takeaway |
|---|---|
| Image classification | Image in, one label from a fixed set out; a building block for detection, captioning, Go |
| Semantic gap | An image is a grid of numbers; meaning isn't visible in them |
| Challenges | Viewpoint, lighting, clutter, occlusion, deformation, intraclass variation, fine-grained classes, context |
| Hand-coding | Rules (holes, edges) don't scale |
| Data-driven approach | Collect data → train → evaluate on new images; `train()` and `predict()` |
| Nearest neighbor | Memorize; copy the label of the closest image (L1 or L2); train O(1), predict O(N) |
| Vectorization | $\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2a^\top b$: all distances in one matrix multiply |
| 🔎 Approximate search | IVF: search a few k-means cells; big speedup, small loss |
| k-NN | Majority vote of k neighbors: smoother boundaries, robust to outliers; ties need a rule |
| L1 vs. L2 | Diamond vs. circle; L1 depends on the axes, L2 doesn't |
| Hyperparameters | Choices about the algorithm; tune on **validation** or cross-validation; test **once** |
| Curse of dimensionality | Exponentially many points needed; distances concentrate |
| Pixels aren't informative | Very different changes can have identical L2 distance |
| Better features | Hand-designed features help a little; data augmentation or learned features help a lot |
| 🔎 Evaluation | Confusion matrices, top-k, and checking for train/test duplicates |

**In CS231n 2026's words:** in image classification we start with a training set of images and labels and must predict labels on the test set; kNN predicts labels from the k nearest training examples; the distance metric and k are hyperparameters; choose them with a validation set; **only run on the test set once, at the very end!**

### Check yourself
<details><summary>1. With N training images, how much work do nearest neighbor's training and prediction take? Why is that backwards?</summary>

Training: O(1) (store the data). Prediction: O(N) (compare with every training image). We'd rather pay at training time, which happens once, than at prediction time, which happens constantly.
</details>

<details><summary>2. Why does k = 1 always score 100% on the training data?</summary>

Every training image is its own nearest neighbor, at distance 0, so it always gets its own label. That makes training accuracy useless for choosing k.
</details>

<details><summary>3. What's wrong with choosing hyperparameters on the test set, if you only report the result once?</summary>

The choice itself used the test set: you picked whichever setting happened to do best on those images, so the reported number is biased upward (§9.2 measured it) and no longer estimates performance on new data.
</details>

<details><summary>4. Why does the no-loop distance code clip with np.maximum(sq, 0)?</summary>

Floating-point rounding can make ||a||² + ||b||² − 2a·b slightly negative when a ≈ b, and the square root of a negative number is nan.
</details>

<details><summary>5. Give two reasons kNN on raw pixels works poorly on natural images.</summary>

It's slow at test time, and pixel distances don't reflect meaning: small shifts or lighting changes produce large distances (or very different changes produce identical ones), and in high dimensions distances concentrate.
</details>

<details><summary>6. Which fix for shifted digits worked best in §12, and why?</summary>

Data augmentation: putting shifted copies in the training set means a shifted test image has a similarly shifted neighbor. The invariance comes from the data rather than from a hand-designed feature.
</details>

### Exercises
1. Implement **distance-weighted kNN** (each neighbor votes with weight $1/d$) and compare it with the majority vote using `cross_validate`.
2. Repeat §9.2 with 5, 60, and 300 settings, and with test-set A sizes of 50, 150, and 500. How does the optimism change?
3. In §7, try `n_cells` = 8, 32, 128 and plot recall@1 against distances computed. Where's the best trade-off?
4. Add **rotation** (±15°) to the test images in §12. Which method handles it? Add rotated copies to the augmented training set.
5. Improve §4's rules for 20 minutes. How many rules did you need, and which digits still fail?
6. (Colab) Run 5-fold cross-validation for k on CIFAR-10 with `knn_torch`. Do you find k ≈ 7, as CS231n does?

### References
* CS231n course notes, *Image Classification* (cs231n.github.io/classification).
* LeCun et al. 1998 (MNIST); Krizhevsky 2009 (CIFAR); Deng et al. 2009 (ImageNet); Zhou et al. 2017 (Places); Lake et al. 2015 (Omniglot).
* Cover & Hart 1967 (nearest-neighbor classification); Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, §2.5.
* Johnson, Douze & Jégou 2017 (FAISS); Jégou, Douze & Schmid 2011 (product quantization).
* Dalal & Triggs 2005 (HOG); Devlin et al. 2015 (nearest-neighbor captioning); Caron et al. 2021 (DINO).
* Recht et al. 2019; Barz & Denzler 2020 (ciFAIR); Northcutt, Athalye & Mueller 2021 (label errors).

**Next:** Lecture 3, Linear Classifiers.